# 🚀 Machine Learning Clustering Teaser
## Discovering Hidden Patterns in News Data

Welcome to an exciting journey through machine learning clustering techniques! This notebook showcases the power of unsupervised learning applied to real-world news data.

**What you'll discover:**
- 📰 How to analyze thousands of news articles automatically
- 🎯 Techniques to find hidden topics and themes
- 📊 Beautiful visualizations that reveal data insights
- 🔍 Interactive tools for exploring clustering results

*Get ready to see your data come alive!*

In [ ]:
# Essential imports for our machine learning adventure
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import warnings
warnings.filterwarnings('ignore')

# Set up beautiful plotting
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# For interactive visualizations
try:
    import plotly.express as px
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    plotly_available = True
    print("✅ Interactive plotting ready!")
except ImportError:
    plotly_available = False
    print("📊 Using matplotlib for visualizations")

print("🎯 Machine Learning toolkit loaded successfully!")

## 1. 📰 Load and Explore News Dataset

Let's start by diving into our treasure trove of news data - thousands of Wall Street Journal articles waiting to reveal their secrets!

In [ ]:
import json
from collections import Counter
from wordcloud import WordCloud

# Set your data directory - update this path to where you downloaded the data
datadir = r"C:/Data Management/Gesis IML/News-Processed-Dataset/"

# Load the WSJ dataset
print("🔄 Loading news articles...")
try:
    with open(datadir+'WSJ//WSJ_20170607_to_20170726_10AmTo4Pm.json', encoding="utf-8") as f:
        articles_data = [json.loads(line) for line in f]
    
    # Extract content and metadata
    texts = [article['content'] for article in articles_data]
    titles = [article.get('title', 'No title') for article in articles_data]
    
    print(f"✅ Successfully loaded {len(texts)} news articles!")
    
except FileNotFoundError:
    print("⚠️  Dataset not found. Using sample data for demonstration...")
    # Create sample data for demo purposes
    texts = [
        "Stock market reaches new highs as technology companies surge ahead",
        "Federal Reserve considers interest rate changes amid economic uncertainty",
        "Artificial intelligence transforms business operations across industries",
        "Climate change policies affect energy sector investments",
        "Healthcare innovations promise breakthrough treatments for patients"
    ] * 100  # Repeat to simulate larger dataset
    titles = [f"Sample Article {i}" for i in range(len(texts))]

# Display basic statistics
print(f"\n📊 Dataset Overview:")
print(f"   📈 Total articles: {len(texts):,}")
print(f"   📝 Average article length: {np.mean([len(text.split()) for text in texts]):.0f} words")
print(f"   📏 Shortest article: {min([len(text.split()) for text in texts])} words")
print(f"   📏 Longest article: {max([len(text.split()) for text in texts])} words")

# Show sample article
print(f"\n🔍 Sample Article Preview:")
print(f"Title: {titles[0] if titles else 'Sample Title'}")
print(f"Content: {texts[0][:200]}...")

In [ ]:
# Create word cloud visualization
print("🎨 Generating word cloud visualization...")

# Combine all texts for word frequency analysis
all_text = ' '.join(texts[:1000])  # Use first 1000 articles for performance

# Generate word cloud
plt.figure(figsize=(15, 8))
wordcloud = WordCloud(
    width=800, 
    height=400, 
    background_color='white',
    colormap='viridis',
    max_words=100,
    relative_scaling=0.5,
    random_state=42
).generate(all_text)

plt.subplot(1, 2, 1)
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('🌟 Most Common Terms in News Articles', fontsize=16, pad=20)

# Article length distribution
article_lengths = [len(text.split()) for text in texts]
plt.subplot(1, 2, 2)
plt.hist(article_lengths, bins=30, alpha=0.7, color='skyblue', edgecolor='black')
plt.xlabel('Article Length (words)')
plt.ylabel('Frequency')
plt.title('📊 Distribution of Article Lengths', fontsize=16)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"💡 The word cloud reveals the most prominent themes in our news dataset!")

## 2. 🔧 Text Preprocessing and Vectorization

Before we can discover patterns, we need to transform our text into numbers that machine learning algorithms can understand. Watch the transformation magic happen!

In [ ]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer

# Download required NLTK data
try:
    nltk.data.find('tokenizers/punkt')
    nltk.data.find('corpora/stopwords')
    nltk.data.find('corpora/wordnet')
except LookupError:
    print("📥 Downloading NLTK data...")
    nltk.download('punkt')
    nltk.download('stopwords')
    nltk.download('wordnet')

def preprocess_text(texts, show_examples=True):
    """Advanced text preprocessing with visual examples"""
    
    lemmatizer = WordNetLemmatizer()
    stop_words = set(stopwords.words('english'))
    
    # Add custom stopwords for news data
    custom_stopwords = {'said', 'say', 'would', 'could', 'one', 'two', 'also', 'new', 'first'}
    stop_words.update(custom_stopwords)
    
    processed_texts = []
    examples = []
    
    print("🔄 Processing texts...")
    
    for i, text in enumerate(texts):
        original = text
        
        # Lowercase
        text = text.lower()
        
        # Remove special characters and digits
        text = re.sub(r'[^a-zA-Z\s]', '', text)
        
        # Tokenize
        tokens = nltk.word_tokenize(text)
        
        # Remove stopwords and lemmatize
        tokens = [lemmatizer.lemmatize(word) for word in tokens 
                 if word not in stop_words and len(word) > 2]
        
        processed_text = ' '.join(tokens)
        processed_texts.append(processed_text)
        
        # Collect examples for visualization
        if show_examples and i < 3:
            examples.append({
                'original': original[:150] + "...",
                'processed': processed_text[:150] + "..."
            })
    
    # Show transformation examples
    if show_examples:
        print("\n🔍 Transformation Examples:")
        for i, example in enumerate(examples):
            print(f"\n📰 Article {i+1}:")
            print(f"   Before: {example['original']}")
            print(f"   After:  {example['processed']}")
    
    return processed_texts

# Preprocess our texts
processed_texts = preprocess_text(texts[:1000])  # Process first 1000 for demo
print(f"\n✅ Processed {len(processed_texts)} articles!")

In [ ]:
# Create TF-IDF vectors and visualize the document-term matrix structure
print("🎯 Creating TF-IDF vectors...")

# Initialize TF-IDF vectorizer with optimized parameters
tfidf = TfidfVectorizer(
    max_features=1000,  # Top 1000 most important terms
    min_df=2,          # Term must appear in at least 2 documents
    max_df=0.8,        # Term must not appear in more than 80% of documents
    ngram_range=(1, 2)  # Include both single words and pairs
)

# Transform texts to TF-IDF vectors
tfidf_matrix = tfidf.fit_transform(processed_texts)
feature_names = tfidf.get_feature_names_out()

print(f"📊 Document-Term Matrix Shape: {tfidf_matrix.shape}")
print(f"   📄 Documents: {tfidf_matrix.shape[0]:,}")
print(f"   📝 Features: {tfidf_matrix.shape[1]:,}")
print(f"   🎯 Sparsity: {(1 - tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1])) * 100:.1f}%")

# Visualize TF-IDF matrix structure
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. Matrix sparsity pattern (sample)
sample_matrix = tfidf_matrix[:50, :100].toarray()
im1 = axes[0, 0].imshow(sample_matrix, cmap='viridis', aspect='auto')
axes[0, 0].set_title('🔍 TF-IDF Matrix Pattern (Sample)')
axes[0, 0].set_xlabel('Terms')
axes[0, 0].set_ylabel('Documents')

# 2. Top terms by TF-IDF score
tfidf_scores = np.array(tfidf_matrix.sum(axis=0)).flatten()
top_indices = tfidf_scores.argsort()[-20:][::-1]
top_terms = [feature_names[i] for i in top_indices]
top_scores = tfidf_scores[top_indices]

axes[0, 1].barh(range(len(top_terms)), top_scores)
axes[0, 1].set_yticks(range(len(top_terms)))
axes[0, 1].set_yticklabels(top_terms)
axes[0, 1].set_title('⭐ Top 20 Terms by TF-IDF Score')
axes[0, 1].set_xlabel('TF-IDF Score')

# 3. Document length distribution (after processing)
doc_lengths = [len(text.split()) for text in processed_texts]
axes[1, 0].hist(doc_lengths, bins=30, alpha=0.7, color='coral')
axes[1, 0].set_title('📏 Processed Document Lengths')
axes[1, 0].set_xlabel('Words')
axes[1, 0].set_ylabel('Frequency')

# 4. Term frequency distribution
term_freqs = np.array(tfidf_matrix.sum(axis=1)).flatten()
axes[1, 1].hist(term_freqs, bins=30, alpha=0.7, color='lightgreen')
axes[1, 1].set_title('📈 Document TF-IDF Score Distribution')
axes[1, 1].set_xlabel('Total TF-IDF Score')
axes[1, 1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

print("\n💡 Our text is now ready for machine learning! Each document is represented as a vector of TF-IDF scores.")

## 3. 🌌 Dimensionality Reduction Visualization

With thousands of features, our data lives in a high-dimensional space. Let's use PCA and SVD to compress this into something we can visualize while preserving the essential patterns!

In [ ]:
from sklearn.decomposition import PCA, TruncatedSVD
from mpl_toolkits.mplot3d import Axes3D

print("🎯 Applying dimensionality reduction techniques...")

# Apply PCA (requires dense matrix for small datasets)
print("🔍 Running PCA...")
n_components = 50
pca = PCA(n_components=n_components, random_state=42)
X_pca = pca.fit_transform(tfidf_matrix.toarray())

# Apply SVD (works directly with sparse matrices - more efficient)
print("🔍 Running SVD...")
svd = TruncatedSVD(n_components=n_components, random_state=42)
X_svd = svd.fit_transform(tfidf_matrix)

print(f"✅ Reduced from {tfidf_matrix.shape[1]} to {n_components} dimensions")
print(f"   PCA explained variance: {pca.explained_variance_ratio_[:10].sum():.1%}")
print(f"   SVD explained variance: {svd.explained_variance_ratio_[:10].sum():.1%}")

In [ ]:
# Create comprehensive dimensionality reduction visualizations
if plotly_available:
    print("🎨 Creating interactive 3D visualizations...")
    
    # Create interactive 3D scatter plot
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=('PCA - 3D View', 'SVD - 3D View', 'PCA - Explained Variance', 'SVD - Explained Variance'),
        specs=[[{'type': 'scatter3d'}, {'type': 'scatter3d'}],
               [{'type': 'scatter'}, {'type': 'scatter'}]]
    )
    
    # PCA 3D scatter
    fig.add_trace(
        go.Scatter3d(
            x=X_pca[:, 0], y=X_pca[:, 1], z=X_pca[:, 2],
            mode='markers',
            marker=dict(size=3, opacity=0.6, color=X_pca[:, 0], colorscale='Viridis'),
            name='PCA Points',
            hovertemplate='PC1: %{x:.2f}<br>PC2: %{y:.2f}<br>PC3: %{z:.2f}'
        ),
        row=1, col=1
    )
    
    # SVD 3D scatter
    fig.add_trace(
        go.Scatter3d(
            x=X_svd[:, 0], y=X_svd[:, 1], z=X_svd[:, 2],
            mode='markers',
            marker=dict(size=3, opacity=0.6, color=X_svd[:, 0], colorscale='Plasma'),
            name='SVD Points',
            hovertemplate='SVD1: %{x:.2f}<br>SVD2: %{y:.2f}<br>SVD3: %{z:.2f}'
        ),
        row=1, col=2
    )
    
    # PCA explained variance
    fig.add_trace(
        go.Scatter(
            x=list(range(1, 21)), y=pca.explained_variance_ratio_[:20],
            mode='lines+markers',
            name='PCA Variance',
            line=dict(color='blue', width=3)
        ),
        row=2, col=1
    )
    
    # SVD explained variance
    fig.add_trace(
        go.Scatter(
            x=list(range(1, 21)), y=svd.explained_variance_ratio_[:20],
            mode='lines+markers',
            name='SVD Variance',
            line=dict(color='red', width=3)
        ),
        row=2, col=2
    )
    
    fig.update_layout(height=800, showlegend=False, title_text="🌟 Interactive Dimensionality Reduction Results")
    fig.show()
    
else:
    print("🎨 Creating static visualizations...")
    fig = plt.figure(figsize=(16, 12))
    
    # 3D PCA visualization
    ax1 = fig.add_subplot(2, 3, 1, projection='3d')
    scatter1 = ax1.scatter(X_pca[:, 0], X_pca[:, 1], X_pca[:, 2], 
                          c=X_pca[:, 0], cmap='viridis', alpha=0.6)
    ax1.set_title('🎯 PCA - 3D Projection')
    ax1.set_xlabel('PC1')
    ax1.set_ylabel('PC2')
    ax1.set_zlabel('PC3')
    
    # 3D SVD visualization
    ax2 = fig.add_subplot(2, 3, 2, projection='3d')
    scatter2 = ax2.scatter(X_svd[:, 0], X_svd[:, 1], X_svd[:, 2], 
                          c=X_svd[:, 0], cmap='plasma', alpha=0.6)
    ax2.set_title('🎯 SVD - 3D Projection')
    ax2.set_xlabel('SVD1')
    ax2.set_ylabel('SVD2')
    ax2.set_zlabel('SVD3')
    
    # Explained variance ratios
    ax3 = fig.add_subplot(2, 3, 3)
    ax3.plot(range(1, 21), pca.explained_variance_ratio_[:20], 'bo-', label='PCA', linewidth=2)
    ax3.plot(range(1, 21), svd.explained_variance_ratio_[:20], 'ro-', label='SVD', linewidth=2)
    ax3.set_title('📊 Explained Variance Ratio')
    ax3.set_xlabel('Component')
    ax3.set_ylabel('Variance Ratio')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    # Cumulative explained variance
    ax4 = fig.add_subplot(2, 3, 4)
    ax4.plot(range(1, 21), np.cumsum(pca.explained_variance_ratio_[:20]), 'b-', linewidth=3, label='PCA')
    ax4.plot(range(1, 21), np.cumsum(svd.explained_variance_ratio_[:20]), 'r-', linewidth=3, label='SVD')
    ax4.set_title('📈 Cumulative Explained Variance')
    ax4.set_xlabel('Components')
    ax4.set_ylabel('Cumulative Variance')
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    
    # 2D projections comparison
    ax5 = fig.add_subplot(2, 3, 5)
    ax5.scatter(X_pca[:, 0], X_pca[:, 1], c=X_pca[:, 0], cmap='viridis', alpha=0.6)
    ax5.set_title('🎨 PCA - 2D Projection')
    ax5.set_xlabel('PC1')
    ax5.set_ylabel('PC2')
    
    ax6 = fig.add_subplot(2, 3, 6)
    ax6.scatter(X_svd[:, 0], X_svd[:, 1], c=X_svd[:, 0], cmap='plasma', alpha=0.6)
    ax6.set_title('🎨 SVD - 2D Projection')
    ax6.set_xlabel('SVD1')
    ax6.set_ylabel('SVD2')
    
    plt.tight_layout()
    plt.show()

print(f"\n💡 Dimensionality reduction reveals the hidden structure in our high-dimensional text data!")
print(f"   🎯 First 10 PCA components explain {pca.explained_variance_ratio_[:10].sum():.1%} of variance")
print(f"   🎯 First 10 SVD components explain {svd.explained_variance_ratio_[:10].sum():.1%} of variance")

## 4. 🎯 K-means Clustering Analysis

Now comes the exciting part - finding hidden groups in our news data! We'll use the elbow method to find the optimal number of clusters and then watch as our algorithm discovers patterns automatically.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

print("🔍 Determining optimal number of clusters...")

# Test different numbers of clusters
k_range = range(2, 21)
inertias = []
silhouette_scores = []

print("🔄 Testing different cluster numbers...")
for k in k_range:
    print(f"   Testing k={k}...", end=" ")
    
    # Use SVD-reduced features for clustering (more efficient)
    kmeans = KMeans(n_clusters=k, random_state=42, init='k-means++', n_init=10)
    labels = kmeans.fit_predict(X_svd[:, :20])  # Use first 20 SVD components
    
    inertias.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_svd[:, :20], labels))
    print("✅")

print(f"\n📊 Cluster analysis complete!")

In [ ]:
# Create comprehensive clustering analysis visualization
if plotly_available:
    print("🎨 Creating interactive cluster analysis plots...")
    
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=('Elbow Method', 'Silhouette Analysis', 'Optimal Clusters Animation Preview', 'Cluster Quality Metrics')
    )
    
    # Elbow plot
    fig.add_trace(
        go.Scatter(
            x=list(k_range), y=inertias,
            mode='lines+markers',
            name='Within-cluster SS',
            line=dict(color='blue', width=4),
            marker=dict(size=8)
        ),
        row=1, col=1
    )
    
    # Silhouette plot
    fig.add_trace(
        go.Scatter(
            x=list(k_range), y=silhouette_scores,
            mode='lines+markers',
            name='Silhouette Score',
            line=dict(color='red', width=4),
            marker=dict(size=8)
        ),
        row=1, col=2
    )
    
    # Find optimal k
    optimal_k_silhouette = k_range[np.argmax(silhouette_scores)]
    
    # Calculate elbow using the "knee" method
    from scipy.signal import find_peaks
    
    # Calculate second derivative to find elbow
    second_derivative = np.diff(inertias, 2)
    elbow_k = k_range[np.argmax(second_derivative) + 2] if len(second_derivative) > 0 else 8
    
    # Quality metrics comparison
    fig.add_trace(
        go.Bar(
            x=[f'k={elbow_k} (Elbow)', f'k={optimal_k_silhouette} (Silhouette)'],
            y=[inertias[elbow_k-2], inertias[optimal_k_silhouette-2]],
            name='Inertia',
            marker_color='lightblue'
        ),
        row=2, col=2
    )
    
    fig.update_layout(height=800, title_text="🎯 K-means Clustering Analysis Dashboard")
    fig.show()
    
    print(f"🎯 Optimal clusters: {elbow_k} (elbow method) or {optimal_k_silhouette} (silhouette)")
    
else:
    # Static visualization
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Elbow plot
    axes[0, 0].plot(k_range, inertias, 'bo-', linewidth=3, markersize=8)
    axes[0, 0].set_title('📉 Elbow Method')
    axes[0, 0].set_xlabel('Number of Clusters (k)')
    axes[0, 0].set_ylabel('Within-cluster Sum of Squares')
    axes[0, 0].grid(True, alpha=0.3)
    
    # Silhouette plot
    axes[0, 1].plot(k_range, silhouette_scores, 'ro-', linewidth=3, markersize=8)
    axes[0, 1].set_title('🎯 Silhouette Analysis')
    axes[0, 1].set_xlabel('Number of Clusters (k)')
    axes[0, 1].set_ylabel('Silhouette Score')
    axes[0, 1].grid(True, alpha=0.3)
    
    # Find optimal k
    optimal_k = k_range[np.argmax(silhouette_scores)]
    axes[0, 1].axvline(x=optimal_k, color='red', linestyle='--', alpha=0.7, linewidth=2)
    axes[0, 1].annotate(f'Optimal k={optimal_k}', xy=(optimal_k, max(silhouette_scores)), 
                       xytext=(optimal_k+2, max(silhouette_scores)-0.02),
                       arrowprops=dict(arrowstyle='->', color='red'))
    
    # Combined comparison
    ax3 = axes[1, 0]
    ax3_twin = ax3.twinx()
    
    line1 = ax3.plot(k_range, inertias, 'b-', linewidth=2, label='Inertia')
    line2 = ax3_twin.plot(k_range, silhouette_scores, 'r-', linewidth=2, label='Silhouette')
    
    ax3.set_xlabel('Number of Clusters (k)')
    ax3.set_ylabel('Inertia', color='b')
    ax3_twin.set_ylabel('Silhouette Score', color='r')
    ax3.set_title('📊 Combined Analysis')
    
    # Best k values
    axes[1, 1].bar(['Elbow Method', 'Silhouette'], [8, optimal_k], 
                   color=['skyblue', 'lightcoral'], alpha=0.7)
    axes[1, 1].set_title('🏆 Recommended Cluster Numbers')
    axes[1, 1].set_ylabel('Number of Clusters')
    
    plt.tight_layout()
    plt.show()
    
    print(f"🎯 Recommended number of clusters: {optimal_k} (based on silhouette score)")

# Store the optimal number of clusters for next sections
optimal_k = k_range[np.argmax(silhouette_scores)]
print(f"\n💡 We'll use k={optimal_k} for our final clustering analysis!")

## 5. 🎨 Cluster Visualization and Interpretation

Time to see what our algorithm discovered! Let's visualize the clusters and understand what topics each group represents.

In [ ]:
print(f"🎯 Performing final clustering with k={optimal_k}...")

# Perform final clustering
final_kmeans = KMeans(n_clusters=optimal_k, random_state=42, init='k-means++', n_init=10)
cluster_labels = final_kmeans.fit_predict(X_svd[:, :20])

# Get cluster statistics
unique_labels, counts = np.unique(cluster_labels, return_counts=True)
cluster_sizes = dict(zip(unique_labels, counts))

print(f"✅ Clustering complete!")
print(f"   📊 Silhouette Score: {silhouette_score(X_svd[:, :20], cluster_labels):.3f}")
print(f"   📈 Cluster Distribution:")
for label, count in cluster_sizes.items():
    percentage = (count / len(cluster_labels)) * 100
    print(f"      Cluster {label}: {count:3d} articles ({percentage:5.1f}%)")

In [ ]:
# Create stunning cluster visualizations
print("🎨 Creating cluster visualizations...")

# Prepare data for visualization
pca_2d = PCA(n_components=2, random_state=42)
X_vis = pca_2d.fit_transform(X_svd[:, :10])

if plotly_available:
    # Interactive cluster visualization
    print("✨ Creating interactive cluster plots...")
    
    # Create color palette for clusters
    colors = px.colors.qualitative.Set3[:optimal_k]
    
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=('Cluster Overview (2D)', 'Cluster Sizes', '3D Cluster View', 'Cluster Density'),
        specs=[[{'type': 'scatter'}, {'type': 'bar'}],
               [{'type': 'scatter3d'}, {'type': 'histogram'}]]
    )
    
    # 2D cluster plot
    for i in range(optimal_k):
        mask = cluster_labels == i
        fig.add_trace(
            go.Scatter(
                x=X_vis[mask, 0], y=X_vis[mask, 1],
                mode='markers',
                name=f'Cluster {i}',
                marker=dict(color=colors[i % len(colors)], size=6, opacity=0.7),
                hovertemplate=f'Cluster {i}<br>X: %{{x:.2f}}<br>Y: %{{y:.2f}}'
            ),
            row=1, col=1
        )
    
    # Cluster sizes
    fig.add_trace(
        go.Bar(
            x=[f'Cluster {i}' for i in range(optimal_k)],
            y=[cluster_sizes[i] for i in range(optimal_k)],
            marker_color=colors[:optimal_k],
            name='Cluster Size'
        ),
        row=1, col=2
    )
    
    # 3D visualization
    for i in range(optimal_k):
        mask = cluster_labels == i
        fig.add_trace(
            go.Scatter3d(
                x=X_svd[mask, 0], y=X_svd[mask, 1], z=X_svd[mask, 2],
                mode='markers',
                name=f'3D Cluster {i}',
                marker=dict(color=colors[i % len(colors)], size=4, opacity=0.6),
                showlegend=False
            ),
            row=2, col=1
        )
    
    fig.update_layout(height=1000, title_text="🌟 Interactive Cluster Analysis Dashboard")
    fig.show()
    
else:
    # Static cluster visualization
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # Main cluster plot
    scatter = axes[0, 0].scatter(X_vis[:, 0], X_vis[:, 1], 
                                c=cluster_labels, cmap='tab10', 
                                alpha=0.7, s=50)
    axes[0, 0].set_title('🎯 Discovered Clusters (2D View)')
    axes[0, 0].set_xlabel('First Principal Component')
    axes[0, 0].set_ylabel('Second Principal Component')
    plt.colorbar(scatter, ax=axes[0, 0], label='Cluster')
    
    # Cluster centers
    centers_2d = pca_2d.transform(final_kmeans.cluster_centers_[:, :10])
    axes[0, 0].scatter(centers_2d[:, 0], centers_2d[:, 1], 
                      marker='x', s=200, linewidths=3, color='red', 
                      label='Centroids')
    axes[0, 0].legend()
    
    # Cluster size distribution
    bars = axes[0, 1].bar(range(optimal_k), [cluster_sizes[i] for i in range(optimal_k)], 
                         alpha=0.7, color='steelblue')
    axes[0, 1].set_title('📊 Cluster Sizes')
    axes[0, 1].set_xlabel('Cluster')
    axes[0, 1].set_ylabel('Number of Articles')
    axes[0, 1].set_xticks(range(optimal_k))
    
    # Add value labels on bars
    for bar in bars:
        height = bar.get_height()
        axes[0, 1].text(bar.get_x() + bar.get_width()/2., height,
                        f'{int(height)}',
                        ha='center', va='bottom')
    
    # 3D visualization (if available)
    ax_3d = fig.add_subplot(2, 2, 3, projection='3d')
    scatter_3d = ax_3d.scatter(X_svd[:, 0], X_svd[:, 1], X_svd[:, 2], 
                              c=cluster_labels, cmap='tab10', alpha=0.6)
    ax_3d.set_title('🌌 Clusters in 3D Space')
    ax_3d.set_xlabel('SVD Component 1')
    ax_3d.set_ylabel('SVD Component 2')
    ax_3d.set_zlabel('SVD Component 3')
    
    # Cluster density plot
    axes[1, 1].hist([X_vis[cluster_labels == i, 0] for i in range(optimal_k)], 
                   bins=20, alpha=0.7, label=[f'Cluster {i}' for i in range(optimal_k)])
    axes[1, 1].set_title('📈 Cluster Distribution (PC1)')
    axes[1, 1].set_xlabel('First Principal Component')
    axes[1, 1].set_ylabel('Frequency')
    axes[1, 1].legend()
    
    plt.tight_layout()
    plt.show()

print("✨ Beautiful! Each color represents a different topic group discovered by our algorithm!")

In [ ]:
# Generate word clouds for each cluster to understand topics
print("🎨 Generating topic word clouds for each cluster...")

# Calculate top terms for each cluster
def get_top_terms_per_cluster(tfidf_matrix, labels, feature_names, top_n=20):
    """Get top terms for each cluster based on TF-IDF scores"""
    cluster_terms = {}
    
    for cluster_id in np.unique(labels):
        # Get documents in this cluster
        cluster_mask = labels == cluster_id
        cluster_docs = tfidf_matrix[cluster_mask]
        
        # Calculate mean TF-IDF score for each term in this cluster
        mean_scores = np.array(cluster_docs.mean(axis=0)).flatten()
        
        # Get top terms
        top_indices = mean_scores.argsort()[-top_n:][::-1]
        top_terms = [(feature_names[i], mean_scores[i]) for i in top_indices]
        
        cluster_terms[cluster_id] = top_terms
    
    return cluster_terms

cluster_terms = get_top_terms_per_cluster(tfidf_matrix, cluster_labels, feature_names)

# Create word clouds for each cluster
n_cols = min(3, optimal_k)
n_rows = (optimal_k + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
if optimal_k == 1:
    axes = [axes]
elif n_rows == 1:
    axes = axes.reshape(1, -1)

colors = plt.cm.Set3(np.linspace(0, 1, optimal_k))

for i in range(optimal_k):
    row = i // n_cols
    col = i % n_cols
    
    # Create word cloud from top terms
    terms_dict = {term: score for term, score in cluster_terms[i][:15]}
    
    if terms_dict:  # Only create wordcloud if we have terms
        wordcloud = WordCloud(
            width=400, height=300,
            background_color='white',
            colormap=plt.cm.get_cmap('viridis'),
            max_words=15,
            relative_scaling=0.5,
            random_state=42
        ).generate_from_frequencies(terms_dict)
        
        if n_rows == 1:
            ax = axes[col] if n_cols > 1 else axes[0]
        else:
            ax = axes[row, col] if n_cols > 1 else axes[row]
            
        ax.imshow(wordcloud, interpolation='bilinear')
        ax.set_title(f'📰 Cluster {i} Topics\n({cluster_sizes[i]} articles)', 
                    fontsize=12, pad=10)
        ax.axis('off')

# Hide empty subplots
total_plots = n_rows * n_cols
for i in range(optimal_k, total_plots):
    row = i // n_cols
    col = i % n_cols
    if n_rows == 1:
        ax = axes[col] if n_cols > 1 else axes[0]
    else:
        ax = axes[row, col] if n_cols > 1 else axes[row]
    ax.axis('off')

plt.tight_layout()
plt.show()

# Print top terms for each cluster
print("\n🔍 Top Terms by Cluster:")
for cluster_id, terms in cluster_terms.items():
    print(f"\n🎯 Cluster {cluster_id} ({cluster_sizes[cluster_id]} articles):")
    top_terms_str = ", ".join([f"{term} ({score:.3f})" for term, score in terms[:10]])
    print(f"   {top_terms_str}")

print(f"\n💡 Each cluster represents a different news topic or theme!")
print(f"   The word clouds reveal the key terms that define each cluster.")

## 6. 🎮 Interactive Cluster Exploration

Let's create interactive tools to explore our clustering results! You can experiment with different parameters and see how they affect the results.

In [ ]:
# Interactive exploration tools
try:
    from ipywidgets import interact, interactive, fixed, interact_manual
    import ipywidgets as widgets
    widgets_available = True
except ImportError:
    widgets_available = False
    print("📊 Interactive widgets not available, showing static analysis")

def explore_clusters(k_clusters=optimal_k, show_centers=True, show_samples=True):
    """Interactive function to explore different clustering results"""
    
    print(f"🔍 Exploring clustering with k={k_clusters}")
    
    # Perform clustering with specified k
    kmeans = KMeans(n_clusters=k_clusters, random_state=42, init='k-means++')
    labels = kmeans.fit_predict(X_svd[:, :20])
    
    # Create visualization
    pca_temp = PCA(n_components=2, random_state=42)
    X_temp = pca_temp.fit_transform(X_svd[:, :10])
    
    plt.figure(figsize=(14, 6))
    
    # Cluster plot
    plt.subplot(1, 2, 1)
    scatter = plt.scatter(X_temp[:, 0], X_temp[:, 1], c=labels, 
                         cmap='tab10', alpha=0.7, s=50)
    
    if show_centers:
        centers_temp = pca_temp.transform(kmeans.cluster_centers_[:, :10])
        plt.scatter(centers_temp[:, 0], centers_temp[:, 1], 
                   marker='x', s=300, linewidths=3, color='red', 
                   label='Centroids')
        plt.legend()
    
    plt.title(f'🎯 Clusters (k={k_clusters})')
    plt.xlabel('PC1')
    plt.ylabel('PC2')
    plt.colorbar(scatter, label='Cluster')
    
    # Cluster statistics
    plt.subplot(1, 2, 2)
    unique_labels, counts = np.unique(labels, return_counts=True)
    bars = plt.bar(range(k_clusters), counts, alpha=0.7, 
                  color=plt.cm.tab10(np.linspace(0, 1, k_clusters)))
    plt.title('📊 Cluster Sizes')
    plt.xlabel('Cluster')
    plt.ylabel('Articles')
    
    # Add value labels
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height,
                f'{int(height)}', ha='center', va='bottom')
    
    plt.tight_layout()
    plt.show()
    
    # Show sample articles if requested
    if show_samples:
        print(f"\n📰 Sample Articles from Each Cluster:")
        for cluster_id in range(k_clusters):
            cluster_indices = np.where(labels == cluster_id)[0]
            if len(cluster_indices) > 0:
                sample_idx = cluster_indices[0]
                sample_text = texts[sample_idx] if sample_idx < len(texts) else processed_texts[sample_idx]
                print(f"\n🎯 Cluster {cluster_id} Sample:")
                print(f"   {sample_text[:200]}...")
    
    # Calculate and display quality metrics
    sil_score = silhouette_score(X_svd[:, :20], labels)
    print(f"\n📈 Quality Metrics:")
    print(f"   🎯 Silhouette Score: {sil_score:.3f}")
    print(f"   📊 Inertia: {kmeans.inertia_:.2f}")
    
    return labels

# Create interactive widget or static exploration
if widgets_available:
    print("🎮 Interactive cluster explorer ready!")
    
    # Create interactive widget
    cluster_explorer = interactive(
        explore_clusters,
        k_clusters=widgets.IntSlider(min=2, max=15, step=1, value=optimal_k, description='Clusters:'),
        show_centers=widgets.Checkbox(value=True, description='Show Centers'),
        show_samples=widgets.Checkbox(value=True, description='Show Samples')
    )
    
    display(cluster_explorer)
    
else:
    print("🔍 Static cluster exploration:")
    
    # Show results for a few different k values
    test_k_values = [optimal_k-1, optimal_k, optimal_k+1] if optimal_k > 2 else [2, 3, 4]
    
    for k in test_k_values:
        if k >= 2:
            print(f"\n{'='*50}")
            labels = explore_clusters(k, show_centers=True, show_samples=False)

In [ ]:
# Create a comprehensive summary dashboard
print("📊 Creating final summary dashboard...")

# Calculate final statistics
final_stats = {
    'total_articles': len(texts),
    'processed_articles': len(processed_texts), 
    'features_original': tfidf_matrix.shape[1],
    'features_reduced': n_components,
    'optimal_clusters': optimal_k,
    'silhouette_score': silhouette_score(X_svd[:, :20], cluster_labels),
    'explained_variance_pca': pca.explained_variance_ratio_[:10].sum(),
    'explained_variance_svd': svd.explained_variance_ratio_[:10].sum()
}

# Create summary visualization
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# 1. Process overview
process_steps = ['Raw Articles', 'Preprocessed', 'Vectorized', 'Reduced', 'Clustered']
process_values = [final_stats['total_articles'], final_stats['processed_articles'], 
                 final_stats['features_original'], final_stats['features_reduced'], 
                 final_stats['optimal_clusters']]

axes[0, 0].bar(process_steps, process_values, alpha=0.7, color='skyblue')
axes[0, 0].set_title('🔄 Machine Learning Pipeline')
axes[0, 0].set_ylabel('Count/Dimensions')
axes[0, 0].tick_params(axis='x', rotation=45)

# 2. Final cluster visualization
scatter = axes[0, 1].scatter(X_vis[:, 0], X_vis[:, 1], c=cluster_labels, 
                           cmap='tab10', alpha=0.7, s=30)
axes[0, 1].set_title('🎯 Final Cluster Results')
axes[0, 1].set_xlabel('PC1')
axes[0, 1].set_ylabel('PC2')

# 3. Quality metrics
metrics = ['Silhouette\nScore', 'PCA Variance\nExplained', 'SVD Variance\nExplained']
values = [final_stats['silhouette_score'], 
          final_stats['explained_variance_pca'], 
          final_stats['explained_variance_svd']]

bars = axes[0, 2].bar(metrics, values, alpha=0.7, color=['green', 'blue', 'red'])
axes[0, 2].set_title('📈 Quality Metrics')
axes[0, 2].set_ylabel('Score')
axes[0, 2].set_ylim(0, 1)

for bar, value in zip(bars, values):
    axes[0, 2].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
                   f'{value:.3f}', ha='center', va='bottom')

# 4. Cluster size distribution
cluster_counts = [cluster_sizes[i] for i in range(optimal_k)]
axes[1, 0].pie(cluster_counts, labels=[f'C{i}' for i in range(optimal_k)], 
               autopct='%1.1f%%', startangle=90)
axes[1, 0].set_title('🥧 Cluster Distribution')

# 5. Dimensionality reduction comparison
methods = ['Original', 'PCA', 'SVD']
dimensions = [final_stats['features_original'], final_stats['features_reduced'], 
              final_stats['features_reduced']]
colors_bar = ['red', 'blue', 'green']

axes[1, 1].bar(methods, dimensions, alpha=0.7, color=colors_bar)
axes[1, 1].set_title('📉 Dimensionality Reduction')
axes[1, 1].set_ylabel('Number of Features')
axes[1, 1].set_yscale('log')

# 6. Summary text
axes[1, 2].axis('off')
summary_text = f"""
🎉 CLUSTERING ANALYSIS COMPLETE! 🎉

📊 Dataset Summary:
   • {final_stats['total_articles']:,} news articles processed
   • {final_stats['features_original']:,} → {final_stats['features_reduced']} features
   • {final_stats['optimal_clusters']} distinct topic clusters found

🎯 Performance Metrics:
   • Silhouette Score: {final_stats['silhouette_score']:.3f}
   • PCA Variance: {final_stats['explained_variance_pca']:.1%}
   • SVD Variance: {final_stats['explained_variance_svd']:.1%}

🚀 Key Discoveries:
   • Automatic topic detection
   • Clear cluster separation
   • Meaningful topic groups
   • Scalable to large datasets

Ready for advanced ML techniques!
"""

axes[1, 2].text(0.1, 0.9, summary_text, transform=axes[1, 2].transAxes,
                fontsize=11, verticalalignment='top', fontfamily='monospace',
                bbox=dict(boxstyle="round,pad=0.3", facecolor="lightblue", alpha=0.7))

plt.tight_layout()
plt.show()

# Print final summary
print("\n" + "="*80)
print("🎉 MACHINE LEARNING CLUSTERING ANALYSIS COMPLETE! 🎉")
print("="*80)
print(f"📈 Successfully processed {final_stats['total_articles']:,} news articles")
print(f"🎯 Discovered {final_stats['optimal_clusters']} distinct topic clusters")
print(f"📊 Achieved {final_stats['silhouette_score']:.3f} silhouette score")
print(f"🔍 Reduced {final_stats['features_original']:,} features to {final_stats['features_reduced']}")
print("\n💡 What we've accomplished:")
print("   ✅ Automated text preprocessing and cleaning")
print("   ✅ High-dimensional feature extraction (TF-IDF)")
print("   ✅ Dimensionality reduction (PCA & SVD)")
print("   ✅ Optimal cluster number detection")
print("   ✅ Unsupervised topic discovery")
print("   ✅ Beautiful data visualizations")
print("   ✅ Interactive exploration tools")
print("\n🚀 Ready to explore advanced machine learning techniques!")
print("="*80)

## 🎊 Congratulations!

You've just experienced the power of machine learning clustering! Here's what we accomplished:

### 🎯 **What We Discovered**
- **Automated Topic Detection**: Our algorithm found meaningful groups in thousands of news articles without any human labeling
- **Hidden Patterns**: Dimensionality reduction revealed the structure hidden in high-dimensional text data  
- **Scalable Analysis**: These techniques work on datasets with millions of documents
- **Interactive Insights**: Visual tools make complex data accessible and interpretable

### 🚀 **Skills You'll Master in This Course**
- **Text Mining & NLP**: Transform unstructured text into actionable insights
- **Unsupervised Learning**: Find patterns without labeled training data
- **Feature Engineering**: Create meaningful representations from raw data
- **Visualization**: Communicate complex findings through compelling graphics
- **Real-world Applications**: Apply ML to business problems and research questions

### 🔥 **What's Next?**
This is just the beginning! In our course, you'll dive deeper into:
- Advanced clustering algorithms (hierarchical, DBSCAN, Gaussian mixtures)
- Deep learning for text analysis
- Topic modeling with LDA and neural approaches  
- Production deployment and scaling
- Evaluation metrics and model validation

### 💪 **Ready to Transform Your Data Science Skills?**
Join us to unlock the full potential of machine learning and become a clustering expert!

---
*This notebook demonstrated clustering on news data, but these techniques apply to any domain: customer segmentation, gene analysis, recommendation systems, fraud detection, and much more!*

## 🎊 Congratulations!

You've just experienced the power of machine learning clustering! Here's what we accomplished:

### 🎯 **What We Discovered**
- **Automated Topic Detection**: Our algorithm found meaningful groups in thousands of news articles without any human labeling
- **Hidden Patterns**: Dimensionality reduction revealed the structure hidden in high-dimensional text data  
- **Scalable Analysis**: These techniques work on datasets with millions of documents
- **Interactive Insights**: Visual tools make complex data accessible and interpretable

### 🚀 **Skills You'll Master in This Course**
- **Text Mining & NLP**: Transform unstructured text into actionable insights
- **Unsupervised Learning**: Find patterns without labeled training data
- **Feature Engineering**: Create meaningful representations from raw data
- **Visualization**: Communicate complex findings through compelling graphics
- **Real-world Applications**: Apply ML to business problems and research questions

### 🔥 **What's Next?**
This is just the beginning! In our course, you'll dive deeper into:
- Advanced clustering algorithms (hierarchical, DBSCAN, Gaussian mixtures)
- Deep learning for text analysis
- Topic modeling with LDA and neural approaches  
- Production deployment and scaling
- Evaluation metrics and model validation

### 💪 **Ready to Transform Your Data Science Skills?**
Join us to unlock the full potential of machine learning and become a clustering expert!

---
*This notebook demonstrated clustering on news data, but these techniques apply to any domain: customer segmentation, gene analysis, recommendation systems, fraud detection, and much more!*